# Aula 2 — Exemplos de código: OpenCV na prática

**Minicurso de Visão Computacional para Robótica** · Prof. Kelson Aires · Liga Acadêmica de Robótica Aplicada (LARA) · VipLab · UFPI

Este notebook reúne os programas da **Apostila de Código da Aula 2**, na mesma ordem dos capítulos. Cada célula é um dos arquivos de `exemplos_aula2.zip`, adaptado para o Colab: no lugar das janelas do OpenCV (`cv2.imshow`), que não funcionam aqui, as imagens aparecem logo abaixo da célula, com a função `mostrar`.

| Capítulo | Assunto |
|---|---|
| 1 | Preparando o ambiente |
| 2 | A imagem como array NumPy |
| 3 | Exibindo e desenhando |
| 4 | Segmentação por cor |
| 5 | Filtros e morfologia |
| 6 | Regiões e contornos |
| 7 | Do pixel ao comando |
| 8 | Vídeo |

**Antes de começar:** envie o arquivo `exemplos_aula2.zip` pelo painel de arquivos (ícone de pasta, à esquerda) e rode as células do capítulo 1. Rode as células em ordem: algumas funções definidas num capítulo são usadas nos seguintes.

---
## 1. Preparando o ambiente

No Colab, Python, OpenCV, NumPy e Matplotlib já vêm instalados. Na sua máquina, crie um ambiente virtual e instale as bibliotecas (apostila, seções 1.1 e 1.2):

```bash
python -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\activate
pip install opencv-python numpy matplotlib
```

### 1.1 Os arquivos da aula
Esta célula descompacta o `exemplos_aula2.zip` e entra na pasta. Depois dela, os caminhos dos programas (`imagens/foto4.jpg`, `videos/video2.mp4`) funcionam como na apostila.

In [ ]:
import os, zipfile

if os.path.basename(os.getcwd()) != 'exemplos_aula2':
    if not os.path.isdir('exemplos_aula2') and os.path.exists('exemplos_aula2.zip'):
        zipfile.ZipFile('exemplos_aula2.zip').extractall('.')
    if os.path.isdir('exemplos_aula2'):
        os.chdir('exemplos_aula2')

if os.path.isdir('imagens'):
    print('Pasta atual:', os.getcwd())
    print('Imagens:', sorted(os.listdir('imagens')))
    print('Vídeos :', sorted(os.listdir('videos')))
else:
    print('Envie exemplos_aula2.zip pelo painel de arquivos (ícone de pasta) e rode esta célula de novo.')

### 1.2 Verificando o ambiente
`01_verifica_ambiente.py`

Mostra a versão de cada biblioteca. No Colab as versões são as do servidor do Google; na sua máquina, as do ambiente virtual.

In [ ]:
import sys
import cv2
import numpy as np
import matplotlib

print('Python    ', sys.version.split()[0])
print('OpenCV    ', cv2.__version__)
print('NumPy     ', np.__version__)
print('Matplotlib', matplotlib.__version__)

### 1.3 A função `mostrar`
Substitui as janelas do OpenCV. Recebe uma imagem (ou uma lista delas) e os títulos, converte as coloridas de BGR para RGB e mostra as de um canal em tons de cinza (`cmap='gray', vmin=0, vmax=255`), como explica o capítulo 3 da apostila.

In [ ]:
import matplotlib.pyplot as plt

def mostrar(imagens, titulos=None, largura=4):
    """Mostra uma ou várias imagens do OpenCV lado a lado, dentro do notebook."""
    if not isinstance(imagens, (list, tuple)):
        imagens, titulos = [imagens], [titulos]
    titulos = titulos or [None] * len(imagens)
    fig, eixos = plt.subplots(1, len(imagens), figsize=(largura * len(imagens), largura))
    eixos = eixos if len(imagens) > 1 else [eixos]
    for eixo, img, titulo in zip(eixos, imagens, titulos):
        if img.ndim == 2:
            eixo.imshow(img, cmap='gray', vmin=0, vmax=255)
        else:
            eixo.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        if titulo:
            eixo.set_title(titulo)
        eixo.axis('off')
    plt.tight_layout()
    plt.show()

No Colab também existe a função `cv2_imshow`, que mostra uma imagem BGR com as cores certas:

```python
from google.colab.patches import cv2_imshow
cv2_imshow(cv2.imread('imagens/foto4.jpg'))
```

Ela só existe no Colab; a função `mostrar` funciona em qualquer notebook.

---
## 2. A imagem como array NumPy
Para o OpenCV, uma imagem é um array do NumPy: uma tabela de números com linhas, colunas e canais.

### 2.1 Lendo uma imagem
`02_lendo_imagem.py`

`cv2.imread` lê o arquivo e devolve um array; se o caminho estiver errado, devolve `None` sem dar erro.

In [ ]:
import cv2

imagem = cv2.imread('imagens/foto4.jpg')

if imagem is None:
    print('Imagem não encontrada: confira o caminho')
else:
    print('Tipo do objeto:', type(imagem))
    print('Dimensões     :', imagem.shape)
    print('Tipo dos dados:', imagem.dtype)
    print('Total de números:', imagem.size)

**O que observar:** `shape` vem na ordem linhas, colunas, canais; `uint8` guarda valores de 0 a 255.

### 2.2 Acessando pixels
`02_acessando_pixels.py`

O pixel $(u, v)$ é `imagem[v, u]`: linha antes da coluna. Os três números vêm na ordem B, G, R.

In [ ]:
imagem = cv2.imread('imagens/foto4.jpg')

# pixel na coluna u = 411 e na linha v = 334 (dentro da bola vermelha)
u, v = 411, 334
azul, verde, vermelho = imagem[v, u]
print('Pixel completo :', imagem[v, u])
print('Azul, verde, vermelho:', azul, verde, vermelho)
print('Só o vermelho  :', imagem[v, u, 2])

# um pixel da parede, para comparar
print('Pixel da parede:', imagem[120, 450])

**O que observar:** na bola o terceiro número (vermelho) é o maior; na parede, os três são altos e parecidos.

### 2.3 Recortes
`02_recortes.py`

Um recorte é uma fatia do array. A documentação do NumPy explica que uma fatia é uma **visão** do original, e não uma cópia.

In [ ]:
imagem = cv2.imread('imagens/foto4.jpg')

# recorte de 72 x 72 pixels em volta da bola vermelha
recorte = imagem[298:370, 375:447]
print('Dimensões do recorte:', recorte.shape)

# o recorte é uma visão da imagem: alterar um altera o outro
recorte[:, :] = (0, 0, 0)
print('Pixel da imagem depois de pintar o recorte:', imagem[334, 411])

# para trabalhar de forma independente, copie
imagem = cv2.imread('imagens/foto4.jpg')
copia = imagem[298:370, 375:447].copy()
copia[:, :] = (0, 0, 0)
print('Pixel da imagem depois de pintar a cópia  :', imagem[334, 411])

**O que observar:** pintar o recorte pintou a imagem original; pintar a cópia, não.

### 2.4 Canais de cor
`02_canais.py`

`cv2.split` separa os canais B, G e R; cada um pode ser visto como uma imagem em tons de cinza.

In [ ]:
imagem = cv2.imread('imagens/foto4.jpg')
azul, verde, vermelho = cv2.split(imagem)

print('Dimensões de cada canal:', vermelho.shape)
mostrar([azul, verde, vermelho], ['Canal azul', 'Canal verde', 'Canal vermelho'])

**O que observar:** a bola vermelha é clara no canal vermelho, mas a parede também é: um canal sozinho não separa a bola.

### 2.5 Estatísticas de uma região
`02_estatisticas.py`

Média e mediana de um quadrado de $15 \times 15$ pixels dentro da bola, e a armadilha do `uint8`.

In [ ]:
import numpy as np

imagem = cv2.imread('imagens/foto8.jpg')

# quadrado de 15 x 15 pixels dentro da bola, centrado em (u, v) = (340, 332)
u, v = 340, 332
amostra = imagem[v - 7:v + 8, u - 7:u + 8]
pixels = amostra.reshape(-1, 3)        # 225 linhas (pixels) por 3 colunas (B, G, R)

print('Forma da amostra :', amostra.shape)
print('Forma após reshape:', pixels.shape)
print('Média por canal  :', np.mean(pixels, axis=0).round(1))
print('Mediana por canal:', np.median(pixels, axis=0))

# a armadilha do uint8: a soma "dá a volta" depois de 255
a = np.uint8(200)
b = np.uint8(100)
print('uint8 200 + 100 =', np.array([a]) + np.array([b]))
print('float 200 + 100 =', float(a) + float(b))

---
## 3. Exibindo e desenhando
Na sua máquina, as imagens aparecem em janelas do OpenCV; no Colab, com o Matplotlib.

### 3.1 Janelas do OpenCV
`03_janela.py`

Este programa só funciona na sua máquina (no Colab, `cv2.imshow` não está disponível). Ele fica aqui como referência:

```python
import cv2

imagem = cv2.imread('imagens/foto4.jpg')

cv2.imshow('Rover', imagem)
tecla = cv2.waitKey(0)          # espera uma tecla; 0 = para sempre
cv2.destroyAllWindows()
print('Código da tecla pressionada:', tecla)
```

Para rodá-lo na sua máquina: `python 03_janela.py`. Apertando **q**, ele imprime o código 113.

### 3.2 Matplotlib
`03_matplotlib.py`

As duas armadilhas do Matplotlib com imagens do OpenCV: cores trocadas (BGR) e máscara colorida (mapa de cores padrão).

In [ ]:
imagem = cv2.imread('imagens/foto4.jpg')
hsv = cv2.cvtColor(imagem, cv2.COLOR_BGR2HSV)
mascara = cv2.inRange(hsv, (170, 120, 50), (179, 255, 255))

fig, eixos = plt.subplots(1, 4, figsize=(14, 4))
eixos[0].imshow(imagem)                                    # errado: BGR
eixos[0].set_title('BGR mostrado direto')
eixos[1].imshow(cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB))   # certo: RGB
eixos[1].set_title('convertido para RGB')
eixos[2].imshow(mascara)                                   # mapa de cores padrão
eixos[2].set_title('máscara sem cmap')
eixos[3].imshow(mascara, cmap='gray', vmin=0, vmax=255)    # tons de cinza
eixos[3].set_title("cmap='gray', vmin=0, vmax=255")
for eixo in eixos:
    eixo.axis('off')
plt.tight_layout()
plt.show()

**O que observar:** na primeira imagem a bola vermelha aparece azul; na terceira, a máscara aparece roxa e amarela.

### 3.3 Desenhando e salvando
`03_desenhando.py`

As funções de desenho recebem pontos como (x, y) = (u, v) e cores em B, G, R. Desenhe sempre numa cópia.

In [ ]:
imagem = cv2.imread('imagens/foto4.jpg')
desenho = imagem.copy()                 # desenhe sempre numa cópia

altura, largura = desenho.shape[:2]
cx = largura // 2
cv2.line(desenho, (cx, 0), (cx, altura - 1), (255, 255, 255), 1)
cv2.rectangle(desenho, (350, 270), (475, 400), (255, 200, 0), 3)
cv2.circle(desenho, (150, 398), 77, (0, 255, 255), 3)
cv2.drawMarker(desenho, (411, 334), (255, 255, 0), cv2.MARKER_CROSS, 30, 3)
cv2.putText(desenho, 'u = 411', (360, 260), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

cv2.imwrite('saida_desenho.png', desenho)
mostrar(desenho, 'Desenho', largura=6)

**O que observar:** o arquivo `saida_desenho.png` aparece no painel de arquivos e pode ser baixado.

---
## 4. Segmentação por cor
Converter para HSV e criar uma máscara com `cv2.inRange`. No OpenCV, o H vai de 0 a 179; S e V, de 0 a 255.

### 4.1 Descobrindo o HSV de uma cor
`04_hsv_de_uma_cor.py`

O truque da documentação do OpenCV: converter um array de um único pixel.

In [ ]:
cores = {'vermelho': [0, 0, 255], 'verde': [0, 255, 0], 'azul': [255, 0, 0],
         'amarelo': [0, 255, 255]}

for nome, bgr in cores.items():
    pixel = np.uint8([[bgr]])
    hsv = cv2.cvtColor(pixel, cv2.COLOR_BGR2HSV)
    print(f'{nome:9s} BGR = {bgr} -> HSV = {hsv[0, 0]}')

### 4.2 O HSV dos objetos da foto
`04_hsv_da_bola.py`

Mediana do HSV em pequenas regiões da foto4.

In [ ]:
imagem = cv2.imread('imagens/foto4.jpg')
hsv = cv2.cvtColor(imagem, cv2.COLOR_BGR2HSV)

pontos = {'bola vermelha': (421, 334), 'bola verde clara': (150, 400),
          'bola verde escura': (380, 520), 'parede': (450, 120), 'chão': (80, 600)}

for nome, (u, v) in pontos.items():
    amostra = hsv[v - 7:v + 8, u - 7:u + 8].reshape(-1, 3)
    h, s, val = np.median(amostra, axis=0)
    print(f'{nome:18s} H = {h:3.0f} | S = {s:3.0f} | V = {val:3.0f}')

**O que observar:** parede e chão têm H "verde", mas saturação muito baixa: por isso a regra exige um S mínimo.

### 4.3 inRange e as duas faixas do vermelho
`04_inrange_vermelho.py`

O vermelho fica perto de 0 **e** de 179: duas faixas, juntadas com `cv2.bitwise_or`.

In [ ]:
imagem = cv2.imread('imagens/foto2.jpg')
hsv = cv2.cvtColor(imagem, cv2.COLOR_BGR2HSV)

tom_baixo_1, tom_alto_1 = np.array([0, 120, 50]), np.array([10, 255, 255])
tom_baixo_2, tom_alto_2 = np.array([170, 120, 50]), np.array([179, 255, 255])

ponta_baixa = cv2.inRange(hsv, tom_baixo_1, tom_alto_1)
ponta_alta = cv2.inRange(hsv, tom_baixo_2, tom_alto_2)
mascara = cv2.bitwise_or(ponta_baixa, ponta_alta)

print('Tipo e forma da máscara:', mascara.dtype, mascara.shape)
print('Valores na máscara     :', np.unique(mascara))
print('Pixels brancos: H 0-10 =', cv2.countNonZero(ponta_baixa),
      '| H 170-179 =', cv2.countNonZero(ponta_alta),
      '| juntas =', cv2.countNonZero(mascara))

mostrar([ponta_baixa, ponta_alta, mascara], ['H 0 a 10', 'H 170 a 179', 'Duas faixas'])

**O que observar:** a máscara pega a bola e também as tampas dos marcadores: a cor sozinha não sabe o que é bola.

#### Experimente
Troque `foto2` por `foto9` na célula acima: na foto9, a mesma bola tem H perto de 0 e quase todos os pixels passam para a faixa de 0 a 10.

### 4.4 Mostrando só o que passou na máscara
`04_bitwise_and.py`

`cv2.bitwise_and(imagem, imagem, mask=mascara)` mantém a cor original só onde a máscara é branca.

In [ ]:
imagem = cv2.imread('imagens/foto5.jpg')
hsv = cv2.cvtColor(imagem, cv2.COLOR_BGR2HSV)
mascara = cv2.bitwise_or(cv2.inRange(hsv, (0, 120, 50), (10, 255, 255)),
                         cv2.inRange(hsv, (170, 120, 50), (179, 255, 255)))

somente_vermelho = cv2.bitwise_and(imagem, imagem, mask=mascara)
mostrar([imagem, mascara, somente_vermelho], ['Original', 'Máscara', 'Somente vermelho'])

### 4.5 O verde
`04_verde.py`

O verde fica no meio da roda: uma faixa basta, e faixas estreitas separam as duas bolas verdes.

In [ ]:
imagem = cv2.imread('imagens/foto4.jpg')
hsv = cv2.cvtColor(imagem, cv2.COLOR_BGR2HSV)

verdes = cv2.inRange(hsv, (35, 120, 40), (85, 255, 255))
verde_clara = cv2.inRange(hsv, (35, 120, 40), (55, 255, 255))
verde_escura = cv2.inRange(hsv, (56, 120, 40), (85, 255, 255))

for nome, m in [('as duas', verdes), ('verde clara', verde_clara), ('verde escura', verde_escura)]:
    print(f'{nome:12s}: {cv2.countNonZero(m):6d} pixels')

mostrar([verdes, verde_clara, verde_escura], ['Verdes (H 35 a 85)', 'Verde clara (H 35 a 55)', 'Verde escura (H 56 a 85)'])

---
## 5. Filtros e morfologia
Suavizar a foto antes da regra de cor e limpar a máscara depois dela.

### 5.1 Suavização
`05_suavizacao.py`

Três filtros e o número de regiões que sobra na máscara do vermelho.

In [ ]:
def mascara_vermelho(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    return cv2.bitwise_or(cv2.inRange(hsv, (0, 120, 50), (10, 255, 255)),
                          cv2.inRange(hsv, (170, 120, 50), (179, 255, 255)))

def regioes(mascara):
    n, _ = cv2.connectedComponents(mascara)
    return n - 1                         # o rótulo 0 é o fundo

imagem = cv2.imread('imagens/foto5.jpg')
versoes = {
    'sem filtro        ': imagem,
    'blur (média 5x5)  ': cv2.blur(imagem, (5, 5)),
    'GaussianBlur 5x5  ': cv2.GaussianBlur(imagem, (5, 5), 0),
    'medianBlur 5      ': cv2.medianBlur(imagem, 5),
}
for nome, img in versoes.items():
    print(nome, '-> regiões na máscara:', regioes(mascara_vermelho(img)))

**O que observar:** a suavização tira os pontinhos do chão de granito; as regiões que sobram são objetos vermelhos de verdade.

### 5.2 Elementos estruturantes
`05_elementos.py`

O elemento estruturante é um pequeno array de zeros e uns.

In [ ]:
for nome, forma in [('MORPH_RECT', cv2.MORPH_RECT), ('MORPH_ELLIPSE', cv2.MORPH_ELLIPSE),
                    ('MORPH_CROSS', cv2.MORPH_CROSS)]:
    elemento = cv2.getStructuringElement(forma, (5, 5))
    print(nome)
    print(elemento)

### 5.3 Erosão, dilatação, abertura e fechamento
`05_abertura_fechamento.py`

Abertura = erosão seguida de dilatação (tira pontinhos); fechamento = dilatação seguida de erosão (tapa buracos).

In [ ]:
imagem = cv2.imread('imagens/foto9.jpg')
suave = cv2.GaussianBlur(imagem, (5, 5), 0)
hsv = cv2.cvtColor(suave, cv2.COLOR_BGR2HSV)
mascara = cv2.bitwise_or(cv2.inRange(hsv, (0, 120, 50), (10, 255, 255)),
                         cv2.inRange(hsv, (170, 120, 50), (179, 255, 255)))

ee_abrir = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
ee_fechar = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))

aberta = cv2.morphologyEx(mascara, cv2.MORPH_OPEN, ee_abrir)
fechada = cv2.morphologyEx(aberta, cv2.MORPH_CLOSE, ee_fechar)
erodida = cv2.erode(mascara, ee_abrir, iterations=3)
dilatada = cv2.dilate(mascara, ee_abrir, iterations=3)

for nome, m in [('crua', mascara), ('erodida (3x)', erodida), ('dilatada (3x)', dilatada),
                ('aberta', aberta), ('aberta e fechada', fechada)]:
    n, _ = cv2.connectedComponents(m)
    print(f'{nome:17s}: {n - 1} regiões, {cv2.countNonZero(m):6d} pixels brancos')

mostrar([mascara, erodida, dilatada, fechada], ['Máscara crua', 'Erodida', 'Dilatada', 'Aberta e fechada'], largura=3.5)

**O que observar:** a erosão encolhe a bola, a dilatação a engorda; abrir e depois fechar limpa sem mudar o tamanho e tapa o buraco do brilho.

#### Experimente
Troque o tamanho do fechamento, `(15, 15)`, por `(5, 5)` e depois por `(41, 41)`, e veja o que acontece com o buraco do brilho e com a borda da bola.

---
## 6. Regiões e contornos
Separar as regiões da máscara e medir área, caixa envolvente e centro.

### 6.1 Componentes conexos
`06_componentes.py`

`connectedComponentsWithStats` numera as regiões e devolve uma tabela com caixa e área de cada uma.

In [ ]:
imagem = cv2.imread('imagens/foto2.jpg')
hsv = cv2.cvtColor(cv2.GaussianBlur(imagem, (5, 5), 0), cv2.COLOR_BGR2HSV)
mascara = cv2.bitwise_or(cv2.inRange(hsv, (0, 120, 50), (10, 255, 255)),
                         cv2.inRange(hsv, (170, 120, 50), (179, 255, 255)))
ee = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
mascara = cv2.morphologyEx(mascara, cv2.MORPH_OPEN, ee)

n, rotulos, stats, centros = cv2.connectedComponentsWithStats(mascara)

print('Número de rótulos (com o fundo):', n)
print('rótulo |   x    y    w    h |  área | centroide')
for i in range(n):
    x, y, w, h, area = stats[i]
    cu, cv = centros[i]
    print(f'{i:6d} | {x:3d}  {y:3d}  {w:3d}  {h:3d} | {area:6d} | ({cu:5.1f}, {cv:5.1f})')

**O que observar:** a linha 0 é o fundo; a bola é a região de maior área entre as outras.

### 6.2 Contornos
`06_contornos.py`

`findContours` procura objetos brancos sobre fundo preto e devolve a lista de contornos.

In [ ]:
imagem = cv2.imread('imagens/foto2.jpg')
hsv = cv2.cvtColor(cv2.GaussianBlur(imagem, (5, 5), 0), cv2.COLOR_BGR2HSV)
mascara = cv2.bitwise_or(cv2.inRange(hsv, (0, 120, 50), (10, 255, 255)),
                         cv2.inRange(hsv, (170, 120, 50), (179, 255, 255)))

contornos, hierarquia = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
print('Contornos encontrados:', len(contornos))

areas = sorted((cv2.contourArea(c) for c in contornos), reverse=True)
print('As cinco maiores áreas:', [round(a) for a in areas[:5]])
print('Pontos do primeiro contorno:', contornos[0].shape)

desenho = imagem.copy()
cv2.drawContours(desenho, contornos, -1, (0, 255, 255), 2)
mostrar(desenho, 'Contornos', largura=6)

### 6.3 O maior contorno: caixa envolvente e centro
`06_maior_contorno.py`

O contorno de maior área, a sua caixa (`boundingRect`) e o seu centroide (`moments`).

In [ ]:
imagem = cv2.imread('imagens/foto2.jpg')
hsv = cv2.cvtColor(cv2.GaussianBlur(imagem, (5, 5), 0), cv2.COLOR_BGR2HSV)
mascara = cv2.bitwise_or(cv2.inRange(hsv, (0, 120, 50), (10, 255, 255)),
                         cv2.inRange(hsv, (170, 120, 50), (179, 255, 255)))
ee_abrir = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
ee_fechar = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
mascara = cv2.morphologyEx(mascara, cv2.MORPH_OPEN, ee_abrir)
mascara = cv2.morphologyEx(mascara, cv2.MORPH_CLOSE, ee_fechar)

contornos, _ = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
maior = max(contornos, key=cv2.contourArea)

x, y, w, h = cv2.boundingRect(maior)
momentos = cv2.moments(maior)
u = momentos['m10'] / momentos['m00']
v = momentos['m01'] / momentos['m00']

print('Área do contorno     :', cv2.contourArea(maior))
print('Caixa envolvente     : x =', x, 'y =', y, 'w =', w, 'h =', h)
print('Centro da caixa      :', (x + w // 2, y + h // 2))
print('Centroide (momentos) :', (round(u, 1), round(v, 1)))

desenho = imagem.copy()
cv2.rectangle(desenho, (x, y), (x + w, y + h), (255, 200, 0), 3)
cv2.drawMarker(desenho, (round(u), round(v)), (255, 255, 0), cv2.MARKER_CROSS, 30, 3)
mostrar(desenho, 'Maior contorno', largura=6)

### 6.4 Juntando tudo: a função encontrar_bola
`encontrar_bola.py`

Suavizar, segmentar, limpar e ficar com o maior contorno. A função devolve `(u, v, h, area)` ou `None` e é usada nos capítulos 7 e 8.

In [ ]:
AREA_MINIMA = 400


def mascara_vermelho(img, s_min=120, v_min=50):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    return cv2.bitwise_or(cv2.inRange(hsv, (0, s_min, v_min), (10, 255, 255)),
                          cv2.inRange(hsv, (170, s_min, v_min), (179, 255, 255)))


def limpar_mascara(mascara, abrir=5, fechar=15):
    ee_abrir = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (abrir, abrir))
    ee_fechar = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (fechar, fechar))
    m = cv2.morphologyEx(mascara, cv2.MORPH_OPEN, ee_abrir)
    return cv2.morphologyEx(m, cv2.MORPH_CLOSE, ee_fechar)


def encontrar_bola(img):
    """Devolve (u, v, h, area) do maior contorno vermelho, ou None."""
    suave = cv2.GaussianBlur(img, (5, 5), 0)
    mascara = limpar_mascara(mascara_vermelho(suave))
    contornos, _ = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contornos:
        return None
    maior = max(contornos, key=cv2.contourArea)
    area = cv2.contourArea(maior)
    if area < AREA_MINIMA:
        return None
    x, y, w, h = cv2.boundingRect(maior)
    return x + w // 2, y + h // 2, h, area


for nome in ['foto1', 'foto2', 'foto5', 'foto8', 'foto9']:
    resultado = encontrar_bola(cv2.imread(f'imagens/{nome}.jpg'))
    if resultado is None:
        print(nome, '-> sem bola')
    else:
        u, v, h, area = resultado
        print(f'{nome} -> u = {u:3d} | v = {v:3d} | h = {h:3d} | área = {area:7.0f}')

**O que observar:** no arquivo da apostila, os testes ficam dentro de `if __name__ == '__main__':`; aqui, no notebook, rodam direto.

---
## 7. Do pixel ao comando
Os erros $e_u = c_x - u$ e $e_h = h_{desejado} - h$, e os comandos de giro $\omega$ e avanço $v$. Aqui $\omega > 0$ significa girar para a esquerda.

### 7.1 Erro e comando
`07_erro_e_comando.py`

Usa a função `encontrar_bola` da célula 6.4.

In [ ]:
KP_GIRO = 1 / 320
KP_AVANCO = 1 / 150
H_DESEJADO = 150


def saturar(valor, limite=1.0):
    return max(-limite, min(limite, valor))


for nome in ['foto2', 'foto5', 'foto8', 'foto9']:
    imagem = cv2.imread(f'imagens/{nome}.jpg')
    cx = imagem.shape[1] // 2
    u, v, h, area = encontrar_bola(imagem)
    erro_u = cx - u
    erro_h = H_DESEJADO - h
    omega = saturar(KP_GIRO * erro_u)
    velocidade = saturar(KP_AVANCO * erro_h)
    print(f'{nome}: e_u = {erro_u:+4d} | e_h = {erro_h:+4d} | '
          f'omega = {omega:+.2f} | v = {velocidade:+.2f}')

**O que observar:** na foto2 a bola está à esquerda ($\omega > 0$); na foto9 está perto demais ($v < 0$, recuar).

---
## 8. Vídeo
Um vídeo é uma sequência de quadros; a 30 quadros por segundo, cada quadro tem cerca de 33 ms para ser processado.

### 8.1 Lendo um vídeo
`08_lendo_video.py`

`cv2.VideoCapture` abre o vídeo; `read()` devolve `(ret, quadro)` e `ret` vira `False` no fim.

In [ ]:
captura = cv2.VideoCapture('videos/video2.mp4')
if not captura.isOpened():
    raise RuntimeError('não consegui abrir o vídeo')

print('Quadros por segundo:', captura.get(cv2.CAP_PROP_FPS))
print('Total de quadros   :', int(captura.get(cv2.CAP_PROP_FRAME_COUNT)))
print('Largura x altura   :', int(captura.get(cv2.CAP_PROP_FRAME_WIDTH)),
      'x', int(captura.get(cv2.CAP_PROP_FRAME_HEIGHT)))

lidos = 0
while True:
    ret, quadro = captura.read()
    if not ret:
        break
    lidos += 1
captura.release()
print('Quadros lidos      :', lidos)

### 8.2 Webcam
`08_webcam.py`

A webcam e as janelas do OpenCV não funcionam no Colab. Na sua máquina, este programa mostra a câmera até você apertar **q**:

```python
import cv2

captura = cv2.VideoCapture(0)          # 0 = primeira câmera do computador

while True:
    ret, quadro = captura.read()
    if not ret:
        break
    cv2.imshow('Webcam', quadro)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

captura.release()
cv2.destroyAllWindows()
```

No Colab, a célula abaixo faz o mesmo laço com um vídeo gravado e mostra alguns quadros.

In [ ]:
captura = cv2.VideoCapture('videos/video3.mp4')
quadros, numero = [], 0
while True:
    ret, quadro = captura.read()
    if not ret:
        break
    if numero % 150 == 0:                 # guarda um quadro a cada 150
        quadros.append(quadro)
    numero += 1
captura.release()
mostrar(quadros, [f'quadro {i * 150}' for i in range(len(quadros))], largura=3)

### 8.3 A bola quadro a quadro
`08_bola_no_video.py`

A função `encontrar_bola` em todos os quadros do `video4`, o mais difícil da aula, medindo o tempo de cada quadro.

In [ ]:
import time

captura = cv2.VideoCapture('videos/video4.mp4')
tempos, com_bola, quadros = [], 0, 0

while True:
    ret, quadro = captura.read()
    if not ret:
        break
    inicio = time.perf_counter()
    resultado = encontrar_bola(quadro)
    tempos.append(1000 * (time.perf_counter() - inicio))
    quadros += 1
    if resultado is not None:
        com_bola += 1
        if quadros % 150 == 1:
            u, v, h, area = resultado
            print(f'quadro {quadros:3d}: u = {u}, h = {h}')

captura.release()
print(f'Bola encontrada em {com_bola} de {quadros} quadros ({100 * com_bola / quadros:.0f}%)')
print(f'Tempo médio por quadro: {sum(tempos) / len(tempos):.1f} ms (orçamento: 33 ms)')

**O que observar:** o tempo medido no Colab é o do servidor do Google; no Rover, que tem um processador menor, meça de novo.

### 8.4 Bônus: a bola desenhada em alguns quadros
`—`

Desenha o resultado de `encontrar_bola` numa cópia de quadros espalhados pelo vídeo, como a apostila sugere no fim do capítulo 8.

In [ ]:
captura = cv2.VideoCapture('videos/video4.mp4')
total = int(captura.get(cv2.CAP_PROP_FRAME_COUNT))
imagens, titulos = [], []
for k in np.linspace(0, total - 1, 4).astype(int):
    captura.set(cv2.CAP_PROP_POS_FRAMES, k)
    ret, quadro = captura.read()
    copia = quadro.copy()
    resultado = encontrar_bola(quadro)
    cv2.line(copia, (320, 0), (320, 639), (255, 255, 255), 1)
    if resultado is not None:
        u, v, h, area = resultado
        cv2.drawMarker(copia, (u, v), (255, 255, 0), cv2.MARKER_CROSS, 30, 3)
        cv2.rectangle(copia, (u - h // 2, v - h // 2), (u + h // 2, v + h // 2), (255, 200, 0), 2)
        titulos.append(f'quadro {k}: e_u = {320 - u:+d}')
    else:
        titulos.append(f'quadro {k}: sem bola')
    imagens.append(copia)
captura.release()
mostrar(imagens, titulos, largura=3.2)

---
## Para continuar

- Na sua máquina, rode os arquivos `.py` de `exemplos_aula2.zip` com `python nome.py`: lá as janelas do OpenCV e a webcam funcionam.
- Para escolher os limites de cor clicando na imagem, use a ferramenta **croma** (apostila, capítulo 9): `seg color -i imagens/foto4.jpg`.